# 02 - MySQL JDBC to Hive and HDFS

Read the seeded MySQL tables and demonstrate two storage approaches:

- Orders: partitioned Parquet files in HDFS with a Hive external-table registration.
- Products: CSV files directly in an HDFS directory.
- Customers: newline-delimited JSON files directly in an HDFS directory.

Run notebook 01 first. This notebook includes its complete Spark, JDBC, Hive and HDFS configuration. Rerunning replaces only this exercise's three output directories. Keep a single writer active and keep MySQL unchanged during extraction.

In [ ]:
import os
import re
import socket
from getpass import getpass

MYSQL_DB = "training"
TABLE_PREFIX = "lake16_"
LAB = "mysql_hdfs_2"
assert all(re.fullmatch(r"[a-z][a-z0-9_]*", value) for value in [MYSQL_DB, TABLE_PREFIX, LAB])
MYSQL_HOST = os.getenv("MYSQL_HOST", "mysql")
MYSQL_PORT = int(os.getenv("MYSQL_PORT", "3306"))
MYSQL_USER = os.getenv("MYSQL_USER", os.getenv("DB_USER", "team"))
MYSQL_PASSWORD = os.getenv("MYSQL_PASSWORD") or os.getenv("DB_PASSWORD") or getpass("MySQL password: ")
JDBC_URL = f"jdbc:mysql://{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DB}?useSSL=false&allowPublicKeyRetrieval=true&serverTimezone=UTC"
JDBC_OPTIONS = {"user": MYSQL_USER, "password": MYSQL_PASSWORD, "driver": "com.mysql.cj.jdbc.Driver", "fetchsize": "100"}
HDFS_URI = os.getenv("WORKSHOP_HDFS_URI", "hdfs://namenode:9000").rstrip("/")
assert HDFS_URI.startswith("hdfs://"), "Choose HDFS, not file:/// or S3."
HIVE_URI = os.getenv("HIVE_METASTORE_URI", "thrift://hive-metastore:9083")
HDFS_ROOT = f"{HDFS_URI}/workshop/16-mysql-hdfs-iceberg/{LAB}"
ORDERS_PATH = HDFS_ROOT + "/orders_parquet"
PRODUCTS_PATH = HDFS_ROOT + "/products_csv"
CUSTOMERS_PATH = HDFS_ROOT + "/customers_json"
HIVE_ORDERS = f"spark_catalog.{LAB}.orders_external_2"
LAKE_ROOT = f"s3://warehouse/iceberg/{LAB}"
print("MySQL source:", MYSQL_DB, "table prefix:", TABLE_PREFIX)
print("HDFS root:", HDFS_ROOT)

## Start Spark with Hive support
Spark talks directly to the Hive metastore. HiveServer2 and YARN are not required. The existing NameNode, DataNode and metastore must be running.

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pathlib import Path
assert SparkSession.getActiveSession() is None, "Stop the previous Spark session and restart this kernel."
jars = Path(os.getenv("SPARK_HOME", "/opt/spark")) / "jars"
assert list(jars.glob("mysql-connector-j-*.jar")), "Use the Spark image with its MySQL JDBC driver."
builder = (SparkSession.builder.appName("mysql-hdfs-" + LAB)
    .master(os.getenv("SPARK_MASTER_URL", "spark://spark-master:7077"))
    .config("spark.driver.host", os.getenv("POD_IP") or socket.gethostbyname(socket.gethostname()))
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.cores.max", "1").config("spark.executor.cores", "1")
    .config("spark.executor.memory", "512m")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.ui.showConsoleProgress", "false")
    .config("spark.redaction.regex", "(?i)secret|password|token|access.?key|credential")
    .config("spark.hadoop.fs.defaultFS", "file:///")
    .config("spark.hadoop.dfs.client.use.datanode.hostname", "true")
    .config("spark.hadoop.dfs.replication", "1"))

builder = (builder.config("spark.hadoop.hive.metastore.uris", HIVE_URI)
    .config("spark.sql.warehouse.dir", HDFS_ROOT + "/managed_warehouse")
    .enableHiveSupport())

spark = builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("Spark:", spark.version, "master:", spark.sparkContext.master)

In [ ]:
# Use the Hadoop client already loaded by Spark; no host CLI or local staging is needed.
j = spark.sparkContext._jvm
hadoop = spark.sparkContext._jsc.hadoopConfiguration()
hdfs = j.org.apache.hadoop.fs.FileSystem.get(j.java.net.URI(HDFS_URI), hadoop)
assert hdfs.getUri().toString().startswith("hdfs://")
print("HDFS:", hdfs.getUri().toString())
assert hdfs.exists(j.org.apache.hadoop.fs.Path("/")), "HDFS is not ready."

## Read the source using JDBC

In [ ]:
# These are full table reads, not CDC or incremental extraction.
products = spark.read.jdbc(JDBC_URL, TABLE_PREFIX + "products_2", properties=JDBC_OPTIONS)
customers = spark.read.jdbc(JDBC_URL, TABLE_PREFIX + "customers_2", properties=JDBC_OPTIONS)
orders = (spark.read.format("jdbc").option("url", JDBC_URL)
    .option("dbtable", TABLE_PREFIX + "orders_2")
    .options(**JDBC_OPTIONS)
    .option("partitionColumn", "order_id").option("lowerBound", "2001")
    .option("upperBound", "2007").option("numPartitions", "2").load())
assert products.count() == 4 and customers.count() == 4 and orders.count() == 6
assert orders.select(F.sum(F.col("quantity") * F.col("unit_price"))).first()[0] == 190
for name, frame, key in [("products", products, "product_id"), ("customers", customers, "customer_id"), ("orders", orders, "order_id")]:
    print(name)
    frame.orderBy(key).show(truncate=False)
print("PASS: JDBC source counts and order value 190.00")

In [ ]:
PRODUCT_SCHEMA = "product_id BIGINT, product_name STRING, category STRING, list_price DECIMAL(12,2)"
CUSTOMER_SCHEMA = "customer_id BIGINT, customer_name STRING, country STRING, email STRING"
ORDER_COLUMNS = ["order_id", "customer_id", "product_id", "quantity", "unit_price", "country"]
PRODUCT_COLUMNS = ["product_id", "product_name", "category", "list_price"]
CUSTOMER_COLUMNS = ["customer_id", "customer_name", "country", "email"]

## Write orders as partitioned Parquet and register a Hive external table

`country=IN`, `country=UK` and `country=US` are HDFS directories. Parquet stores the nonpartition columns; Hive records the partition values and their locations. Creating the external table registers existing files. `MSCK REPAIR TABLE` discovers the directory partitions.

In [ ]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS spark_catalog.{LAB} LOCATION '{HDFS_ROOT}/hive_database'")
if spark.catalog.tableExists(HIVE_ORDERS):
    details = spark.sql(f"DESCRIBE EXTENDED {HIVE_ORDERS}").collect()
    actual_location = next(r.data_type for r in details if r.col_name == "Location")
    assert actual_location.rstrip("/") == ORDERS_PATH, "Existing table uses a different location; choose a fresh LAB."
orders.select(*ORDER_COLUMNS).write.mode("overwrite").partitionBy("country").parquet(ORDERS_PATH)
spark.sql(f"""CREATE EXTERNAL TABLE IF NOT EXISTS {HIVE_ORDERS} (
    order_id BIGINT, customer_id BIGINT, product_id BIGINT, quantity INT, unit_price DECIMAL(12,2))
    PARTITIONED BY (country STRING) STORED AS PARQUET LOCATION '{ORDERS_PATH}'""")
spark.sql(f"MSCK REPAIR TABLE {HIVE_ORDERS}")
spark.catalog.refreshTable(HIVE_ORDERS)
spark.sql(f"SHOW PARTITIONS {HIVE_ORDERS}").show(truncate=False)
spark.sql(f"SELECT country, count(*) AS orders, sum(quantity*unit_price) AS amount FROM {HIVE_ORDERS} GROUP BY country ORDER BY country").show()

## Write products directly to HDFS as CSV

There is no Hive table for this dataset. Spark writes a directory containing part files and a success marker. The header, delimiter, quoting, escaping and null marker are part of the read/write contract. A quoted comma belongs to the product name, not another column.

In [ ]:
(products.select(*PRODUCT_COLUMNS).write.mode("overwrite")
    .option("header", True).option("escape", '"').option("nullValue", "\\N")
    .csv(PRODUCTS_PATH))
products_back = (spark.read.schema(PRODUCT_SCHEMA).option("header", True)
    .option("escape", '"').option("nullValue", "\\N").option("mode", "FAILFAST").csv(PRODUCTS_PATH))
products_back.orderBy("product_id").show(truncate=False)

## Write customers directly to HDFS as JSON

Each line is a JSON object. Keep null fields visible in the files. Read with an explicit schema so repeated reads preserve the same types even when a column contains nulls.

In [ ]:
(customers.select(*CUSTOMER_COLUMNS).write.mode("overwrite")
    .option("ignoreNullFields", "false").json(CUSTOMERS_PATH))
customers_back = spark.read.schema(CUSTOMER_SCHEMA).option("mode", "FAILFAST").json(CUSTOMERS_PATH)
customers_back.orderBy("customer_id").show(truncate=False)

## Compare each HDFS result with MySQL and inspect real files

Both directions of `exceptAll` check values and duplicate counts. The external table must be registered as EXTERNAL and show exactly three country partitions. Products and customers remain directory-only datasets.

In [ ]:
orders_back = spark.table(HIVE_ORDERS).select(*ORDER_COLUMNS)
for source, target, columns in [(orders,orders_back,ORDER_COLUMNS), (products,products_back,PRODUCT_COLUMNS), (customers,customers_back,CUSTOMER_COLUMNS)]:
    assert source.select(*columns).exceptAll(target.select(*columns)).count() == 0
    assert target.select(*columns).exceptAll(source.select(*columns)).count() == 0
assert {r.partition for r in spark.sql(f"SHOW PARTITIONS {HIVE_ORDERS}").collect()} == {"country=IN", "country=UK", "country=US"}
details = spark.sql(f"DESCRIBE EXTENDED {HIVE_ORDERS}").collect()
assert next(r.data_type for r in details if r.col_name == "Type") == "EXTERNAL"
for path in [ORDERS_PATH, PRODUCTS_PATH, CUSTOMERS_PATH]:
    print("FILES", path)
    iterator = hdfs.listFiles(j.org.apache.hadoop.fs.Path(path), True)
    while iterator.hasNext():
        file = iterator.next()
        print(file.getPath().toString(), file.getLen())
print("PASS: Hive external Parquet, HDFS CSV and HDFS JSON exactly match MySQL")

## Release Spark
Run this cell before starting the next notebook. Use a fresh kernel for the next notebook.

In [ ]:
spark.stop()